# CottonLens: ilk 16 adaylık ablation birimi

Bu defter, mevcut `research-free-data-v1` deneyinin **yalnız base feature / XGBoost / T+1 / fold 1** kararını çalıştırır. Tamamlanan altı fitlik pilotu doğrulanmış cache üzerinden kullanır. Geri kalan 127 karar birimi ve yeni veri kaynakları otomatik başlamaz. **Tümünü çalıştır** bir birim tamamlayıp durur.

İlerleme çubuğu kalıcı fit sayısını gösterir (`14/167` başlangıç, `167/167` beklenen bitiş); zaman yüzdesi değildir. Karar ve sonuç aynı deney klasörüne kaydedilir. Bu fold daha önce görülen tarihsel araştırma verisidir; sonucuna bakarak başarı ilan edilmez.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CottonLensAI')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT = 'research-free-data-v1'  # Retain this identifier for resumes of EXACTLY this data/source.


In [ ]:
import hashlib, importlib, json, sys, zipfile
from pathlib import PurePosixPath
SOURCE_ZIP = DRIVE_ROOT / 'sources/cottonlens-free-research-source-v1.zip'
EXPECTED_SOURCE_ID = 'c669ca9a9d4db2647b80993881efda2ba65b050ea26c6fd16a06a5c1e5dd1b93'
EXPECTED_ZIP_SHA256 = 'ff4eec811ef9549e7311a467475cca2b7826044566aecf5704b792668b06c6b1'
expected_zip = SOURCE_ZIP.with_suffix('.zip.sha256').read_text().split()[0]
assert expected_zip == EXPECTED_ZIP_SHA256, 'Source ZIP sidecar differs from pinned bundle'
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == EXPECTED_ZIP_SHA256, 'Source ZIP checksum mismatch'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    source = json.loads(archive.read('.source-manifest.json'))
    payload = {key: value for key, value in source.items() if key != 'source_id'}
    identity = hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    assert identity == source['source_id'] == EXPECTED_SOURCE_ID, 'Source manifest mismatch'
    assert len(archive.namelist()) == len(set(archive.namelist())), 'Duplicate archive members'
    assert set(archive.namelist()) == set(source['files']) | {'.source-manifest.json'}, 'Unexpected archive files'
    REPO = Path('/content') / ('cottonlens-source-' + identity[:16])
    REPO.mkdir(exist_ok=True)
    for name in archive.namelist():
        member = PurePosixPath(name)
        assert not member.is_absolute() and '..' not in member.parts and chr(92) not in name and ':' not in name
        content = archive.read(name)
        if name in source['files']:
            assert hashlib.sha256(content).hexdigest() == source['files'][name], name
        target = REPO / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert target.read_bytes() == content, f'Existing Colab source changed: {name}'
        else:
            with target.open('xb') as handle:
                handle.write(content)
sys.path.insert(0, str(REPO / 'ml'))
import colab_setup
importlib.reload(colab_setup)
colab_setup.PROJECT = REPO
from colab_setup import setup, run, INFERENCE
print('Source identity:', identity, 'Git HEAD:', source['git_head'], 'Includes working-tree edits:', source['working_tree_dirty'])


In [ ]:
PYTHON = setup(research=True)  # Locked managed Python 3.12.11; no global packages.
print('Training interpreter:', PYTHON)
print('Inference-only interpreter:', INFERENCE / 'bin/python')


In [ ]:
import json
from colab_setup import process_env

assert identity == EXPECTED_SOURCE_ID, 'Wrong source bundle; use the pinned free-research ZIP'
folder = DRIVE_ROOT / 'experiments' / EXPERIMENT
ready_path = folder / 'ready.json'
assert ready_path.is_file(), 'Prepared experiment missing; do not create a new experiment here'
ready = json.loads(ready_path.read_text(encoding='utf-8'))
assert ready['identity']['source_id'] == identity, 'Experiment and source bundle differ'
assert ready['identity']['profile'] == 'free-data-v1', 'Wrong experiment profile'
assert len(ready['identity']['split']['folds']) == 8, 'Expected eight frozen historical folds'
assert not ready['identity']['publication_sources'], 'Unexpected new data package: review before training'
assert not (folder / '.writer-lock').exists(), 'Writer lock present: verify old Colab session stopped before recovery'
BASE = [PYTHON, '-m', 'cottonlens_ml.research.engine', '--repo', REPO,
        '--drive-root', DRIVE_ROOT, '--experiment', EXPERIMENT]
run([*BASE, '--stage', 'status'])  # Read-only inventory; no model fit.
print('Pinned source and prepared cohort match. Next cell runs the eight small diagnostic fits.')


In [ ]:
import json
import colab_progress
from IPython.display import display, Markdown

assert json.loads((folder / 'diagnosis.json').read_text(encoding='utf-8'))['status'] == 'passed'
PILOT = folder / 'pilots' / 'base-xgboost-t1-fold1-candidate1.json'
assert PILOT.exists(), 'Run the six-fit pilot first; its completed fits will be reused'
completed_dir = folder / 'ledger' / 'completed'
saved = len(list(completed_dir.glob('*.json')))
assert 14 <= saved <= 167, f'Unexpected completed-fit count {saved}; inspect the experiment before continuing'
panel = display(Markdown('Starting only the first 16-candidate ablation decision unit.'), display_id=True)
def update(message):
    panel.update(Markdown('```text\n' + message + '\n```'))
unit_program = "import json\nimport sys\nimport time\nfrom pathlib import Path\n\nfrom cottonlens_ml.research.engine import Experiment\nfrom cottonlens_ml.research.ledger import read_record\nfrom cottonlens_ml.research.search import recipes\nfrom cottonlens_ml.runtime_guard import require_colab_training\nfrom cottonlens_ml.sprint import writer\n\nrequire_colab_training()\nroot, repo = Path(sys.argv[1]), Path(sys.argv[2])\nexperiment = Experiment(root, repo=repo)\nassert experiment.identity['profile'] == 'free-data-v1'\nassert read_record(root / 'diagnosis.json')['status'] == 'passed'\nfold = experiment.identity['split']['folds'][0]\nassert fold['fold'] == 1\nnamespace = 'ablate-base'\ncandidates = recipes('xgboost', 1, experiment.identity['groups']['base'], 16)\nstarted = time.monotonic()\nwith writer(root):\n    decision = experiment.tune('xgboost', 1, fold, candidates, namespace)\n    outcome = experiment.outer(decision['chosen'], fold, namespace)\nprint('UNIT1 completed in', round(time.monotonic() - started, 1), 'seconds', flush=True)\nprint('UNIT1 result:', json.dumps({'family': 'xgboost', 'horizon': 1,\n      'group': 'base', 'fold': 1, 'candidates': len(candidates),\n      'chosen_inner_relative_mae': decision['chosen']['score'],\n      'outer_mae': outcome['metrics']['mae'],\n      'outer_relative_mae_improvement_pct': outcome['metrics']['relative_mae_improvement_pct'],\n      'historical_evidence_only': True}), flush=True)\n"
colab_progress.supervise(
    [PYTHON, '-c', unit_program, str(folder), str(REPO)],
    cwd=REPO, env=process_env(), drive_logs=DRIVE_ROOT / 'reports',
    decisions=completed_dir, decision_pattern='*.json', total=167, update=update,
)
decision_path = folder / 'decisions' / 'ablate-base' / 'xgboost-t1-fold1.json'
outer_path = folder / 'outer' / 'ablate-base' / 'xgboost-t1-fold1.json'
assert decision_path.is_file() and outer_path.is_file(), 'Ablation decision or outer result missing'
decision = json.loads(decision_path.read_text(encoding='utf-8'))
outer = json.loads(outer_path.read_text(encoding='utf-8'))
print('Chosen inner relative MAE:', decision['chosen']['score'])
print('Outer fold-1 MAE:', outer['metrics']['mae'])
print('Outer fold-1 relative MAE improvement %:', outer['metrics']['relative_mae_improvement_pct'])
print('These are reused historical development results, not a holdout or model acceptance.')
run([*BASE, '--stage', 'status'])
